# 02 · Denetim sayfaları (Adım 6)

**Ayarlar:** sağ panel → *Add Input* → `melihkotman/disasterm3-turkiye`; *Accelerator: None*; *Internet: On*; *Secrets*: `GITHUB_TOKEN` **Attached**.

Örneklem (`data_manifest/denetim/denetim_orneklem_v1.csv`, 96 karo) görüntülere bakılmadan önce üretildi ve değiştirilmez. Bu defter her karo için öncesi/sonrası sayfa üretir; sayfada şehir/event yazmaz (körleme).
Bitince *Output* sekmesinden `tum_sayfalar.zip` ve `dogrulama_sayfalari.zip` indirilir (çıktı ~100 MB).

In [ ]:
import os, subprocess, time, glob, shutil
REPO_URL = "github.com/MelihKotman/Deprem-VLM-RAG.git"
if not os.path.exists("/tmp/repo"):
    tok = None
    for i in range(6):                       # Kaggle Secrets servisi 429 verebilir: bekleyip tekrar dene
        try:
            from kaggle_secrets import UserSecretsClient
            tok = UserSecretsClient().get_secret("GITHUB_TOKEN"); break
        except Exception as e:
            print(f"secrets denemesi {i+1}/6 başarısız ({type(e).__name__}); {60*(i+1)} sn bekleniyor"); time.sleep(60*(i+1))
    if tok:
        r = subprocess.run(["git","clone","-q",f"https://{tok}@{REPO_URL}","/tmp/repo"],capture_output=True,text=True)
        if r.returncode != 0: raise SystemExit("git clone başarısız (repo adı/token?)")
        subprocess.run(["git","-C","/tmp/repo","remote","set-url","origin",f"https://{REPO_URL}"],check=True)
    else:                                     # YEDEK: Add Input ile eklenmiş kod dataset'i (kod.zip içeriği)
        hit = glob.glob("/kaggle/input/*/src/audit_pages.py") + glob.glob("/kaggle/input/*/*/src/audit_pages.py")
        if not hit: raise SystemExit("Secrets 429 ve yedek kod dataset'i yok: docs/ veya sohbetteki yedek yolu uygula")
        shutil.copytree(os.path.dirname(os.path.dirname(hit[0])), "/tmp/repo")
os.chdir("/tmp/repo")
!pip -q install -r requirements.txt
!python -m src.config

## 1. Bir kez: Adım 5'ten devreden doğrulama (repo manifest'i ile dataset birebir mi?)

In [ ]:
!python -m src.dm3_verify

## 2. Örneklem sağlam mı? (96 karo, dosyaların hepsi var mı)

In [ ]:
import pandas as pd
s = pd.read_csv("data_manifest/denetim/denetim_orneklem_v1.csv")
print(len(s), "karo;", s.dogrulama.sum(), "doğrulama alt kümesi")
print(s.groupby(["bolge","dur"]).size().unstack(fill_value=0))

## 3. Sayfaları üret (önce 2 karoluk deneme, sonra hepsi; `--quadrants` öncesinin 2x yakın çeyreklerini de üretir)

In [ ]:
!python -m src.audit_pages --out /kaggle/working/denetim --only A001,A002

In [ ]:
!python -m src.audit_pages --out /kaggle/working/denetim --quadrants
!ls /kaggle/working/denetim && du -sh /kaggle/working/denetim/*.zip

Sonra: *Save Version → Save & Run All (Commit)*; bitince sürümün *Output* sekmesinden `tum_sayfalar.zip` ve `dogrulama_sayfalari.zip` + `sayfa_ozet.csv` indir (`RAGEarthQuake/denetim/` altına koy; repo dışı).